<a href="https://colab.research.google.com/github/nikhilmooloo02-stack/CLARIX_AI_AGENT/blob/main/CLARIX_AI_AGENT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# CELL 1 - Install dependencies
!pip install anthropic gradio chromadb pypdf2 pillow requests -q
print("All packages installed")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.5/837.5 kB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 71.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 105.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 70.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6

In [2]:
# CELL 2 - Connect to Claude AI
import anthropic
from google.colab import userdata

# Get API key securely
client = anthropic.Anthropic(api_key=userdata.get('ANTHROPIC_API_KEY'))

# Test connection
def test_connection():
    print(client.messages.create(
      model="claude-sonnet-4-5",
      max_tokens=100,
      messages=[{"role": "user", "content": "Say: HI MY NAME IS CLARIX, YOUR AI ASSISTANT. HOW CAN I HELP YOU TODAY?."}]
    ).content[0].text)

test_connection()


HI MY NAME IS CLARIX, YOUR AI ASSISTANT. HOW CAN I HELP YOU TODAY?


In [3]:
SYSTEM_PROMPT = """You are CLARIX, a professional AI customer support agent
for South African SMEs. Your job is to:
- Handle customer complaints with empathy
- Provide clear solutions and timelines
- Escalate serious issues when necessary
- Always acknowledge the customer's frustration first
- Be concise and professional"""

conversation_history = []

def ask_clarix(user_message):
    conversation_history.append({
        "role": "user",
        "content": user_message
    })

    response = client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=500,
        system=SYSTEM_PROMPT,
        messages=conversation_history
    )

    reply = response.content[0].text
    conversation_history.append({
        "role": "assistant",
        "content": reply
    })

    print(f"\nCLARIX: {reply}\n")

# Test it
ask_clarix("A customer says their order arrived damaged and wants a refund.")


CLARIX: I understand how frustrating it is to receive a damaged order. I sincerely apologize for this experience.

**Here's how I can help you right now:**

1. **Immediate refund option**: I can process a full refund to your original payment method within 5-7 business days

2. **Replacement option**: If you'd prefer, I can send a replacement with express shipping at no charge, arriving within 2-3 business days

**I'll need from you:**
- Photo of the damaged item (helps us improve packaging)
- Your order number

**Next steps:**
- Once you provide the details, I'll process this immediately
- You won't need to return the damaged item

Which option works better for you - refund or replacement? And could you please share your order number so I can get this resolved for you today?



In [4]:
# Cell 4 — Install Excel reader
!pip install openpyxl pandas -q
print("Excel reader ready")


Excel reader ready


In [7]:
# Cell 5 — Load ShaNeal Product List into CLARIX
import pandas as pd
import chromadb
from google.colab import files

# Step 1 — Upload your Excel file
print("Please upload your Excel product list...")
uploaded = files.upload()

# Step 2 — Read the Excel file
filename = list(uploaded.keys())[0]
df = pd.read_excel(filename)

print(f"Found {len(df)} products")
print(f"Columns detected: {list(df.columns)}")
print(df.head(3))  # Shows first 3 rows so you can verify it loaded correctly


Please upload your Excel product list...


Saving ShaNeal Products.xlsx to ShaNeal Products.xlsx
Found 26102 products
Columns detected: ['This XML file does not appear to have any style information associated with it. The document tree is shown below.']
  This XML file does not appear to have any style information associated with it. The document tree is shown below.
0  <urlset xmlns:image="http://www.google.com/sch...                                                               
1                                              <url>                                                               
2             <loc>https://shanealonline.co.za</loc>                                                               


In [20]:
# Cell 4 — CLARIX Gradio UI
import gradio as gr

def chat(message, history):
    conversation = []
    for human, assistant in history:
        conversation.append({"role": "user", "content": human})
        conversation.append({"role": "assistant", "content": assistant})

    conversation.append({"role": "user", "content": message})

    response = client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=500,
        system=SYSTEM_PROMPT,
        messages=conversation
    )

    return response.content[0].text

demo = gr.ChatInterface(
    fn=chat,
    title="CLARIX — AI Customer Support Agent",
    description="Professional AI-powered customer support for your business.",
    examples=[
        "My order arrived damaged and I want a refund.",
        "I've been waiting 3 weeks and nobody is responding.",
        "I want to cancel my order immediately.",
    ],
    theme=gr.themes.Soft()
)

demo.launch(share=True)

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://59e77b6863c71fde64.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [8]:
# Cell 6 - Scraping tools
!pip install requests beautifulsoup4 pandas openpyxl -q
print("Scraping tools ready")

Scraping tools ready


In [19]:
# Cell 7+8 — Scrape directly from ShaNeal's live sitemap
import requests
from bs4 import BeautifulSoup
import time

# Step 1 — Fetch the sitemap directly from the website
print("Fetching sitemap from ShaNealonline.co.za...")
sitemap_url = "https://shanealonline.co.za/sitemap.xml"
headers = {'User-Agent': 'Mozilla/5.0'}

response = requests.get(sitemap_url, headers=headers, timeout=15)
print(f"Status: {response.status_code}")

soup = BeautifulSoup(response.text, 'xml')
all_urls = []

for loc in soup.find_all('loc'):
    url = loc.get_text(strip=True)
    if 'product-detail' in url:
        all_urls.append(url)

print(f"✅ Found {len(all_urls)} product URLs")

# Step 2 — Scrape first 5 to test
def scrape_product(url):
    try:
        response = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(response.text, 'html.parser')
        name = soup.find('h1')
        name = name.get_text(strip=True) if name else None
        price = soup.select_one('.price, [class*="price"]')
        price = price.get_text(strip=True) if price else None
        return {'name': name, 'price': price}
    except:
        return {'name': None, 'price': None}

print("\nScraping first 5 products to test...\n")
for i, url in enumerate(all_urls[:5]):
    result = scrape_product(url)
    print(f"{i+1}. {result['name']} | {result['price']}")
    time.sleep(1)


Fetching sitemap from ShaNealonline.co.za...
Status: 200
✅ Found 0 product URLs

Scraping first 5 products to test...



In [ ]:
!git clone https://github.com/nikhilmooloo02-stack/CLARIX_AI_AGENT.git

Cloning into 'CLARIX_AI_AGENT'...
remote: Enumerating objects: 6, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 6 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (6/6), done.
